# 01_04: Named Entity Recognition (NER)

**Objective:** Learn to extract named entities (people, organizations, locations, etc.) from text using token classification models.

**Prerequisites:** Basic Python, familiarity with HuggingFace pipelines (Notebook 00_03).

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min RAM | Recommended Setup | Notes |
|-------------|------------|------|---------|-------------------|-------|
| **Small (CPU)** | dslim/bert-base-NER | ~430MB | 4GB RAM | Any CPU | Default for this notebook |
| **Large (GPU)** | Jean-Baptiste/camembert-ner | ~440MB | 8GB RAM | RTX 4080+ | French NER model |

## Expected Behaviors

### First Time Running
- **Model download**: ~430MB for BERT-NER (1-3 minutes)
- Cached in `~/.cache/huggingface/hub/` for subsequent runs

### What You'll See
- Models identify entities like persons (PER), organizations (ORG), locations (LOC), and miscellaneous (MISC)
- Subword tokens get merged back into full entity spans
- Confidence scores indicate model certainty for each entity

### Common Observations
- Well-known entities (Apple, Google, New York) get high confidence
- Ambiguous entities ("Apple" as fruit vs company) depend on context
- Nested or overlapping entities are challenging for standard NER models

## Overview

**Named Entity Recognition (NER)** is a token classification task where each token in the input is assigned a label. Unlike text classification (which labels the whole sequence), NER labels individual tokens.

### NER Tag Schemes

The most common tagging scheme is **BIO** (Beginning-Inside-Outside):

| Tag | Meaning | Example |
|-----|---------|--------|
| `B-PER` | Beginning of a Person entity | **B-PER**: "Barack" |
| `I-PER` | Inside (continuation) of a Person entity | **I-PER**: "Obama" |
| `B-ORG` | Beginning of an Organization | **B-ORG**: "Google" |
| `B-LOC` | Beginning of a Location | **B-LOC**: "Paris" |
| `B-MISC` | Beginning of Miscellaneous entity | **B-MISC**: "English" |
| `O` | Outside any entity | "the", "is", "a" |

For example: "Barack Obama visited Google in Paris"
```
Barack  Obama  visited  Google  in  Paris
B-PER   I-PER  O        B-ORG   O   B-LOC
```

## Setup and Installation

We import all libraries, set the random seed for reproducibility, and check the hardware environment.

In [ ]:
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForTokenClassification,
)
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Version metadata
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Model Selection

We use `dslim/bert-base-NER`, a BERT model fine-tuned on the CoNLL-2003 dataset for English NER. It recognizes four entity types: PER, ORG, LOC, and MISC.

In [ ]:
# Option 1: Small Model (CPU-friendly, recommended for beginners)
MODEL_NAME = 'dslim/bert-base-NER'          # ~430MB, 4 entity types

# Option 2: Large Model (GPU-optimized, more entity types)
# MODEL_NAME = 'Jean-Baptiste/camembert-ner'  # ~440MB, French NER
# MODEL_NAME = 'xlm-roberta-large-finetuned-conll03-english'  # ~1.2GB

## Method 1: Pipeline API

The simplest way to run NER is with the `pipeline` function. It handles tokenization, inference, and post-processing (including merging subword tokens back into complete entities) automatically.

In [ ]:
# Create NER pipeline
ner_pipeline = pipeline(
    'ner',
    model=MODEL_NAME,
    aggregation_strategy='simple',  # Merge subword tokens into entities
    device=device,
)

# Test with a sample sentence
text = 'Barack Obama was born in Honolulu, Hawaii and served as the 44th President of the United States.'
entities = ner_pipeline(text)

print(f'Input: "{text}"\n')
print(f'Found {len(entities)} entities:\n')

entity_df = pd.DataFrame([{
    'Entity': e['word'],
    'Type': e['entity_group'],
    'Score': f'{e["score"]:.4f}',
    'Start': e['start'],
    'End': e['end'],
} for e in entities])
entity_df

The `aggregation_strategy='simple'` parameter is crucial — without it, you'd get individual subword predictions (e.g., "Bar", "##ack", "Obama" separately). The pipeline merges them into coherent entity spans.

### Aggregation Strategies

| Strategy | Behavior |
|----------|----------|
| `'none'` | Raw token-level predictions (includes `##` subword prefixes) |
| `'simple'` | Merge adjacent tokens with same entity type |
| `'first'` | Use first token's score for the merged entity |
| `'average'` | Average scores across merged tokens |
| `'max'` | Take maximum score across merged tokens |

In [ ]:
# Compare aggregation strategies
text_aggr = 'Elon Musk founded SpaceX in Hawthorne, California.'

print('=== aggregation_strategy="none" (raw tokens) ===')
raw_pipeline = pipeline('ner', model=MODEL_NAME, aggregation_strategy='none', device=device)
raw_entities = raw_pipeline(text_aggr)
for e in raw_entities:
    print(f'  {e["word"]:15s} → {e["entity"]:8s} ({e["score"]:.4f})')

print(f'\n=== aggregation_strategy="simple" (merged) ===')
merged_entities = ner_pipeline(text_aggr)
for e in merged_entities:
    print(f'  {e["word"]:20s} → {e["entity_group"]:8s} ({e["score"]:.4f})')

## Method 2: Manual Model Loading

For more control over the NER process, we can load the model and tokenizer separately. This lets us inspect token-level predictions, apply custom post-processing, and understand exactly what the model outputs.

In [ ]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

# Check label mapping
id2label = model.config.id2label
label2id = model.config.label2id
print('Label mapping:')
for label_id, label_name in sorted(id2label.items()):
    print(f'  {label_id} → {label_name}')

In [ ]:
def extract_entities_manual(
    text: str,
    tokenizer: transformers.PreTrainedTokenizer,
    model: torch.nn.Module,
    id2label: dict[int, str],
    confidence_threshold: float = 0.5,
) -> list[dict[str, object]]:
    """Extract named entities with manual model inference.

    Args:
        text: Input text to analyze.
        tokenizer: The tokenizer for the NER model.
        model: The token classification model.
        id2label: Mapping from label IDs to label names.
        confidence_threshold: Minimum confidence to include an entity.

    Returns:
        List of entity dictionaries with word, type, score, and position.
    """
    # Tokenize with offset mapping to track character positions
    inputs = tokenizer(
        text,
        return_tensors='pt',
        return_offsets_mapping=True,
        truncation=True,
    )
    offset_mapping = inputs.pop('offset_mapping')[0]
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Forward pass
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get predictions
    probabilities = torch.softmax(outputs.logits, dim=-1)
    predictions = torch.argmax(probabilities, dim=-1)[0]
    scores = probabilities[0].max(dim=-1).values
    
    # Collect entities (skip special tokens and 'O' labels)
    entities: list[dict[str, object]] = []
    current_entity: dict[str, object] | None = None
    
    for idx, (pred, score, offset) in enumerate(zip(
        predictions, scores, offset_mapping
    )):
        label = id2label[pred.item()]
        conf = score.item()
        start, end = offset.tolist()
        
        # Skip special tokens (offset 0,0 for [CLS], [SEP])
        if start == 0 and end == 0:
            if current_entity:
                entities.append(current_entity)
                current_entity = None
            continue
        
        if label.startswith('B-') and conf >= confidence_threshold:
            if current_entity:
                entities.append(current_entity)
            current_entity = {
                'word': text[start:end],
                'type': label[2:],
                'score': conf,
                'start': start,
                'end': end,
            }
        elif label.startswith('I-') and current_entity and label[2:] == current_entity['type']:
            current_entity['word'] = text[current_entity['start']:end]
            current_entity['end'] = end
            current_entity['score'] = min(current_entity['score'], conf)
        else:
            if current_entity:
                entities.append(current_entity)
                current_entity = None
    
    if current_entity:
        entities.append(current_entity)
    
    return entities


text_manual = 'Tim Cook announced that Apple will open a new office in London next year.'
entities_manual = extract_entities_manual(text_manual, tokenizer, model, id2label)

print(f'Input: "{text_manual}"\n')
pd.DataFrame(entities_manual)

The manual approach gives us full control over the entity extraction pipeline. We can customize the confidence threshold, handle edge cases in entity merging, and access the raw probability distributions for each token. This is essential for production systems where you need fine-grained control.

## Visualizing NER Results

Visualizing entities in context helps understand model behavior. Let's build a simple text annotation display and an entity type distribution chart.

In [ ]:
def display_entities(
    text: str,
    entities: list[dict[str, object]],
) -> None:
    """Print text with entity annotations highlighted.

    Args:
        text: Original input text.
        entities: List of entity dictionaries from NER extraction.
    """
    # Sort entities by start position
    sorted_entities = sorted(entities, key=lambda e: e['start'])
    
    print('Annotated text:')
    print('-' * 70)
    
    last_end = 0
    annotated = ''
    for entity in sorted_entities:
        annotated += text[last_end:entity['start']]
        entity_text = entity.get('word', text[entity['start']:entity['end']])
        annotated += f'[{entity_text}]({entity["type"]})'
        last_end = entity['end']
    annotated += text[last_end:]
    print(annotated)
    print('-' * 70)


# Demo
demo_text = (
    'Sundar Pichai, CEO of Google, met with Emmanuel Macron in Paris '
    'to discuss European AI regulations at the Elysee Palace.'
)
demo_entities = ner_pipeline(demo_text)
display_entities(demo_text, demo_entities)
print()
pd.DataFrame([{
    'Entity': e['word'],
    'Type': e['entity_group'],
    'Score': f'{e["score"]:.4f}',
} for e in demo_entities])

In [ ]:
def plot_entity_distribution(
    entities_list: list[list[dict[str, object]]],
    labels: list[str],
) -> None:
    """Plot distribution of entity types across multiple texts.

    Args:
        entities_list: List of entity lists (one per text).
        labels: Text labels for the legend.
    """
    all_types = set()
    for entities in entities_list:
        for e in entities:
            all_types.add(e.get('entity_group', e.get('type', 'UNK')))
    all_types = sorted(all_types)
    
    counts = []
    for entities in entities_list:
        type_count = {t: 0 for t in all_types}
        for e in entities:
            entity_type = e.get('entity_group', e.get('type', 'UNK'))
            type_count[entity_type] += 1
        counts.append(type_count)
    
    x = np.arange(len(all_types))
    width = 0.8 / len(labels)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (label, count) in enumerate(zip(labels, counts)):
        values = [count[t] for t in all_types]
        ax.bar(x + i * width, values, width, label=label[:30])
    
    ax.set_xlabel('Entity Type', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Entity Type Distribution', fontsize=13)
    ax.set_xticks(x + width * (len(labels) - 1) / 2)
    ax.set_xticklabels(all_types)
    ax.legend()
    plt.tight_layout()
    plt.show()


# Analyze entity distributions across different texts
texts_for_analysis = [
    'Apple CEO Tim Cook and Microsoft CEO Satya Nadella met in Silicon Valley to discuss AI partnerships.',
    'The European Union announced new regulations. Germany and France led the initiative in Brussels.',
    'Roger Federer defeated Rafael Nadal at Wimbledon. The Swiss champion won in straight sets.',
]

all_entities = [ner_pipeline(t) for t in texts_for_analysis]
plot_entity_distribution(
    all_entities,
    ['Tech news', 'Politics', 'Sports'],
)

## Practical Applications

NER is a building block for many real-world applications. Let's explore a few common use cases that demonstrate how NER fits into larger systems.

In [ ]:
def analyze_document(
    text: str,
    ner_pipe: transformers.Pipeline,
) -> pd.DataFrame:
    """Extract and summarize all entities in a document.

    Args:
        text: Document text to analyze.
        ner_pipe: HuggingFace NER pipeline.

    Returns:
        DataFrame with unique entities grouped by type.
    """
    entities = ner_pipe(text)
    
    # Group by type and deduplicate
    grouped: dict[str, set[str]] = {}
    for e in entities:
        entity_type = e['entity_group']
        if entity_type not in grouped:
            grouped[entity_type] = set()
        grouped[entity_type].add(e['word'])
    
    rows: list[dict[str, str]] = []
    for entity_type, names in sorted(grouped.items()):
        rows.append({
            'Entity Type': entity_type,
            'Count': str(len(names)),
            'Entities': ', '.join(sorted(names)),
        })
    
    return pd.DataFrame(rows)


# Application 1: News article analysis
news_article = (
    'Tesla CEO Elon Musk announced plans to build a new Gigafactory in '
    'Austin, Texas. The $5 billion investment was welcomed by Governor '
    'Greg Abbott. Tesla already operates factories in Fremont, California '
    'and Shanghai, China. The Austin facility will produce the Cybertruck '
    'and Model Y. SpaceX, another Musk company, recently launched a '
    'Starship from its Boca Chica facility in Texas.'
)

print('=== News Article Entity Analysis ===')
print(f'Article length: {len(news_article.split())} words\n')
analyze_document(news_article, ner_pipeline)

In [ ]:
# Application 2: Batch processing with confidence filtering
def batch_ner_with_filtering(
    texts: list[str],
    ner_pipe: transformers.Pipeline,
    min_confidence: float = 0.9,
) -> pd.DataFrame:
    """Run NER on multiple texts with confidence filtering.

    Args:
        texts: List of input texts.
        ner_pipe: HuggingFace NER pipeline.
        min_confidence: Minimum confidence threshold.

    Returns:
        DataFrame with high-confidence entities from all texts.
    """
    all_results: list[dict[str, str]] = []
    
    for i, text in enumerate(texts):
        entities = ner_pipe(text)
        for e in entities:
            if e['score'] >= min_confidence:
                all_results.append({
                    'Text #': str(i + 1),
                    'Entity': e['word'],
                    'Type': e['entity_group'],
                    'Score': f'{e["score"]:.4f}',
                })
    
    return pd.DataFrame(all_results) if all_results else pd.DataFrame()


batch_texts = [
    'Jeff Bezos stepped down as Amazon CEO. Andy Jassy took over in Seattle.',
    'The WHO declared the pandemic over. Tedros Adhanom spoke at the Geneva headquarters.',
    'LeBron James signed with the Los Angeles Lakers for $154 million.',
]

print('=== Batch NER (confidence >= 0.9) ===')
batch_ner_with_filtering(batch_texts, ner_pipeline, min_confidence=0.9)

## Performance Benchmarking

Let's measure NER inference speed and compare the pipeline approach to manual loading.

In [ ]:
def benchmark_ner(
    texts: list[str],
    ner_pipe: transformers.Pipeline,
    num_runs: int = 5,
) -> dict[str, float]:
    """Benchmark NER pipeline latency.

    Args:
        texts: List of texts to process.
        ner_pipe: HuggingFace NER pipeline.
        num_runs: Number of runs for averaging.

    Returns:
        Dictionary with timing statistics.
    """
    # Warmup
    ner_pipe(texts[0])
    
    times: list[float] = []
    for _ in range(num_runs):
        start = time.perf_counter()
        for text in texts:
            ner_pipe(text)
        elapsed = time.perf_counter() - start
        times.append(elapsed)
    
    return {
        'total_ms': np.mean(times) * 1000,
        'per_text_ms': np.mean(times) * 1000 / len(texts),
        'std_ms': np.std(times) * 1000,
    }


bench_texts = [
    'Elon Musk visited the Tesla factory in Berlin, Germany.',
    'The United Nations met in New York to discuss climate change.',
    'Lionel Messi scored twice for Inter Miami against Atlanta United.',
    'Microsoft acquired Activision Blizzard for $68.7 billion.',
    'Prime Minister Rishi Sunak met President Biden at the White House.',
]

bench_results = benchmark_ner(bench_texts, ner_pipeline)

bench_df = pd.DataFrame([{
    'Model': MODEL_NAME,
    'Texts Processed': len(bench_texts),
    'Total Time (ms)': f'{bench_results["total_ms"]:.1f}',
    'Per Text (ms)': f'{bench_results["per_text_ms"]:.1f}',
    'Device': str(device),
}])
bench_df

## Exercises

1. **Confidence analysis**: Run NER on a paragraph with ambiguous entities (e.g., "Apple" as company vs fruit). Plot the confidence scores and see if the model handles disambiguation correctly.

2. **Custom threshold**: Modify `extract_entities_manual()` to accept different confidence thresholds. How does the number of extracted entities change between 0.3, 0.5, 0.7, and 0.9?

3. **Entity linking**: After extracting entities, try to group them by type and count frequencies. Which entity types appear most often in news articles vs scientific papers?

4. **Multi-sentence NER**: Process a longer document (5+ sentences) and visualize how entity density varies across sentences.

## Key Takeaways

- **NER is token classification**: Each token gets a label (B-PER, I-ORG, O, etc.) using the BIO tagging scheme
- **Aggregation matters**: Use `aggregation_strategy='simple'` in the pipeline to merge subword tokens into complete entity spans
- **Confidence filtering**: Production systems should filter entities by confidence score to reduce false positives
- **Context-dependent**: The same word can be classified differently depending on context ("Apple" as ORG vs "apple" as not an entity)
- **Building block**: NER is a foundation for information extraction, knowledge graphs, and document understanding systems

## Next Steps & Resources

**Next notebook**: [01_05 — Question Answering](01_05_nlp_question_answering.ipynb) — extract answers from text given a question.

**Resources:**
- [HuggingFace NER Tutorial](https://huggingface.co/docs/transformers/tasks/token_classification)
- [CoNLL-2003 Dataset](https://huggingface.co/datasets/conll2003) — Standard NER benchmark
- [Token Classification Models on Hub](https://huggingface.co/models?pipeline_tag=token-classification)
- [Understanding BIO Tagging](https://en.wikipedia.org/wiki/Inside%E2%80%93outside%E2%80%93beginning_(tagging))